# 01 — Data Exploration

Explore ViDia2Std, ViLexNorm, and the reversed corpus.
- Dataset statistics (counts, lengths, splits)
- Region distribution
- Dialect marker frequency analysis
- Sample examples for each task

In [ ]:
import json
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

sns.set_theme(style="whitegrid")

DATA_DIR = Path("../data/processed")

In [ ]:
def load_jsonl(path):
    records = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            records.append(json.loads(line))
    return records

train = load_jsonl(DATA_DIR / "train.jsonl")
dev = load_jsonl(DATA_DIR / "dev.jsonl")
test = load_jsonl(DATA_DIR / "test.jsonl")

print(f"Train: {len(train)}, Dev: {len(dev)}, Test: {len(test)}")

## Task Distribution

In [ ]:
task_counts = Counter(r["task"] for r in train)
print("Task distribution (train):")
for task, count in task_counts.most_common():
    print(f"  {task}: {count}")

fig, ax = plt.subplots(figsize=(10, 5))
tasks = list(task_counts.keys())
counts = [task_counts[t] for t in tasks]
ax.barh(tasks, counts, color=sns.color_palette("viridis", len(tasks)))
ax.set_xlabel("Count")
ax.set_title("Training Data — Task Distribution")
plt.tight_layout()
plt.savefig("../results/figures/task_distribution.png", dpi=150)
plt.show()

## Region Distribution (ViDia2Std)

In [ ]:
dialect_records = [r for r in train if r["task"] == "dialect2std"]
region_counts = Counter(r["region"] for r in dialect_records)
print("Region distribution (dialect2std, train):")
for region, count in region_counts.most_common():
    print(f"  {region}: {count}")

fig, ax = plt.subplots(figsize=(6, 4))
ax.pie(region_counts.values(), labels=region_counts.keys(), autopct="%1.1f%%",
       colors=sns.color_palette("Set2", len(region_counts)))
ax.set_title("ViDia2Std Region Distribution")
plt.savefig("../results/figures/region_distribution.png", dpi=150)
plt.show()

## Sentence Length Distribution

In [ ]:
src_lens = [len(r["source"].split()) for r in train]
tgt_lens = [len(r["target"].split()) for r in train]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(src_lens, bins=50, color="steelblue", alpha=0.7)
axes[0].set_title("Source Length (syllables)")
axes[0].set_xlabel("Length")
axes[1].hist(tgt_lens, bins=50, color="coral", alpha=0.7)
axes[1].set_title("Target Length (syllables)")
axes[1].set_xlabel("Length")
plt.tight_layout()
plt.savefig("../results/figures/length_distribution.png", dpi=150)
plt.show()

print(f"Source: mean={sum(src_lens)/len(src_lens):.1f}, max={max(src_lens)}")
print(f"Target: mean={sum(tgt_lens)/len(tgt_lens):.1f}, max={max(tgt_lens)}")

## Dialect Marker Frequency by Region

In [ ]:
from src.model.config import EvalConfig

markers = EvalConfig().dialect_markers

for region, region_markers in markers.items():
    region_records = [r for r in dialect_records if r["region"] == region]
    print(f"\n--- {region.upper()} ({len(region_records)} samples) ---")
    marker_freq = Counter()
    for r in region_records:
        text_lower = r["source"].lower()
        for m in region_markers:
            if m in text_lower:
                marker_freq[m] += 1
    for m, count in marker_freq.most_common():
        pct = count / max(len(region_records), 1) * 100
        print(f"  {m}: {count} ({pct:.1f}%)")

## Sample Examples

In [ ]:
for task in ["dialect2std", "std2dialect_central", "std2dialect_south", "lexnorm"]:
    samples = [r for r in train if r["task"] == task][:5]
    if not samples:
        continue
    print(f"\n{'='*60}")
    print(f"Task: {task}")
    print(f"{'='*60}")
    for r in samples:
        print(f"  SRC: {r['source']}")
        print(f"  TGT: {r['target']}")
        print()